In [ ]:
## python modules used within this notebook
%matplotlib widget
import numpy as np
from scipy import integrate
from scipy import interpolate
import matplotlib.pyplot as plt
import matplotlib.animation
import matplotlib.colors as colors
import os
import h5py
import sys
import mynumerics as mn
import units
import HHG
from IPython.display import display, Markdown
from IPython.display import HTML


matplotlib.rcParams['animation.embed_limit'] = 200.



# import matplotlib.pyplot as plt
# import matplotlib.colors as colors

# %matplotlib inline

## TDSE with a custom input

We show the interface for the TDSE solver accessed directly through Python. We use this solver for a custom field we define, and then analyse the result in details. We will show the spectrum of the source term, wavefunction, we do energetic analyses via the Gabor transform and [invariant energetic distribution](https://doi.org/10.1103/PhysRevA.106.053115). Finally, we will show the depletion of the ground state.


First, we import the compiled dynamical library and its Pythonic wrapper:

In [ ]:
from PythonTDSE import *


path_to_DLL = os.path.join(
    os.environ['TDSE_1D_BUILD'],
    'libsingleTDSE.so'
)

DLL = TDSE_DLL(path_to_DLL)

In [ ]:
omega0 = mn.ConvertPhoton(1000e-9, 'lambdaSI', 'omegaau')
chirp = 2e-4
E_0 = 0.15  # peak electric field amplitude

T0 = mn.ConvertPhoton(omega0, 'omegaau', 'T0au')
T_max = 3 * T0
N_t = 10000

tgrid = np.linspace(0, T_max, N_t)
E = E_0 * np.sin(np.pi * tgrid / T_max)**2 * np.cos(omega0 * tgrid + chirp * tgrid**2)

gas = 'Ar'
trg_a = 1.1893  # Argon

In [ ]:
inputs1 = inputs_def()

inputs1.init_default_inputs(
    Eguess=-HHG.Ip_list[gas],
    trg_a = HHG.soft_Coulomb_a[gas],
    dt=0.125,
    dx=0.4,
    num_r=16000,
    writewft=1,
    tprint=1.,
    x_int=2.,
    absorber={'type': 0}
)

print('Version 1: xmax =', 0.5 * inputs1.num_r * inputs1.dx)

inputs1.init_time_and_field(DLL, E=E, t=tgrid)
DLL.init_GS(inputs1)

outputs1 = outputs_def()
DLL.call1DTDSE(inputs1, outputs1)

In [ ]:
inputs2 = inputs_def()

inputs2.init_default_inputs(
    Eguess=-HHG.Ip_list[gas],
    trg_a = HHG.soft_Coulomb_a[gas],
    dt=0.125,
    dx=0.4,
    num_r=300,
    writewft=1,
    tprint=1.,
    x_int=2.,
    absorber={
        'type': 1,
        'x_cap': 50.,
        'alpha': 0.001
    }
)

print('Version 2: xmax =', 0.5 * inputs2.num_r * inputs2.dx)

inputs2.init_time_and_field(DLL, E=E, t=tgrid)
DLL.init_GS(inputs2)

outputs2 = outputs_def()
DLL.call1DTDSE(inputs2, outputs2)

In [ ]:
inputs3 = inputs_def()

inputs3.init_default_inputs(
    Eguess=-HHG.Ip_list[gas],
    trg_a = HHG.soft_Coulomb_a[gas],
    dt=0.125,
    dx=0.4,
    num_r=750,
    writewft=1,
    tprint=1.,
    x_int=2.,
    absorber={
        'type': 2,
        'x_cap': 140.
    }
)

print('Version 3: xmax =', 0.5 * inputs3.num_r * inputs3.dx)

inputs3.init_time_and_field(DLL, E=E, t=tgrid)
DLL.init_GS(inputs3)

outputs3 = outputs_def()
DLL.call1DTDSE(inputs3, outputs3)

In [ ]:
TDSEs = [
    {
        'inputs': inputs1,
        'outputs': outputs1,
    },
    {
        'inputs': inputs2,
        'outputs': outputs2,
    },
    {
        'inputs': inputs3,
        'outputs': outputs3,
    },
]

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.colors as colors


omega_max_plot = 3.5  # [a.u.]

# Optional spatial filtering
filter_xgrid = False
x_max_plot = 250.1  # [a.u.]

# Optional filtering of small wavefunction values
filter_wavefunction_min = False
wavefunction_min = 1e-8
wavefunction_max = 0.5


# -------------------------------------------------------------------------
# 1. Electric field
# -------------------------------------------------------------------------

fig1, ax1 = plt.subplots(figsize=(8, 5))

ax1.plot(
    TDSEs[0]['outputs'].get_tgrid(),
    TDSEs[0]['outputs'].get_Efield(),
    label='Electric field'
)

ax1.set_xlabel(r'$t~[\mathrm{a.u.}]$')
ax1.set_ylabel(r'$\mathcal{E}~[\mathrm{a.u.}]$')
ax1.legend()

fig1.tight_layout()
plt.show()


# -------------------------------------------------------------------------
# 2. Harmonic spectra
# -------------------------------------------------------------------------

fig2, ax2 = plt.subplots(figsize=(8, 5))
photon_energy_ranges = []

for k, tdse in enumerate(TDSEs):
    ogrid = tdse['outputs'].get_omegagrid()
    ko_max = mn.FindInterval(ogrid, omega_max_plot)

    photon_energy = mn.ConvertPhoton(ogrid[:ko_max], 'omegaau', 'eV')
    spectrum = np.abs(tdse['outputs'].get_Fsourceterm())[:ko_max]

    ax2.semilogy(photon_energy, spectrum, label=f'Version {k + 1}')
    photon_energy_ranges.append((photon_energy.min(), photon_energy.max()))

ax2.set_xlim(
    min(limits[0] for limits in photon_energy_ranges),
    max(limits[1] for limits in photon_energy_ranges)
)

ax2.set_xlabel(r'$\omega~[\mathrm{eV}]$')
ax2.set_ylabel(
    r'$|(\partial \hat{\jmath}/\partial t)(\omega)|'
    r'~[\mathrm{arb.~u.}]$'
)
ax2.legend()

fig2.tight_layout()
plt.show()


# -------------------------------------------------------------------------
# 3. Wavefunctions
# -------------------------------------------------------------------------

wavefunction_data = []

for tdse in TDSEs[:3]:
    t_psi, x_grid, wavefunction = tdse['outputs'].get_wavefunction(
        tdse['inputs'],
        grids=True
    )

    x_range = np.abs(x_grid) < x_max_plot if filter_xgrid else slice(None)
    psi_plot = np.abs(wavefunction).T[x_range]

    if filter_wavefunction_min:
        psi_plot = np.maximum(psi_plot, wavefunction_min)

    wavefunction_data.append((t_psi, x_grid[x_range], psi_plot))


# Common limits covering all three plots
t_min = min(t_psi.min() for t_psi, x_grid, psi_plot in wavefunction_data)
t_max = max(t_psi.max() for t_psi, x_grid, psi_plot in wavefunction_data)
x_min = min(x_grid.min() for t_psi, x_grid, psi_plot in wavefunction_data)
x_max = max(x_grid.max() for t_psi, x_grid, psi_plot in wavefunction_data)

wavefunction_norm = colors.LogNorm(
    vmin=wavefunction_min,
    vmax=wavefunction_max
)

fig3, axes = plt.subplots(
    1,
    3,
    figsize=(10, 5.5),
    sharex=True,
    sharey=True,
    layout='constrained'
)

for k, (ax, (t_psi, x_grid, psi_plot)) in enumerate(
    zip(axes, wavefunction_data)
):
    pc = ax.pcolormesh(
        t_psi,
        x_grid,
        psi_plot,
        cmap='jet',
        norm=wavefunction_norm,
        shading='auto'
    )

    ax.set_title(f'Version {k + 1}')
    ax.set_xlabel(r'$t~[\mathrm{a.u.}]$')

axes[0].set_ylabel(r'$x~[\mathrm{a.u.}]$')
axes[0].set_xlim(t_min, t_max)
axes[0].set_ylim(x_min, x_max)

cbar = fig3.colorbar(
    pc,
    ax=axes,
    orientation='horizontal',
    location='bottom',
    pad=0.08,
    shrink=0.8,
    aspect=45
)

cbar.set_label(r'$|\psi|~[\mathrm{a.u.}]$')

plt.show()